In [3]:
import pandas as pd
import numpy as np
import math

In [8]:
bb=pd.read_excel("fantasy_last3yr.xlsx",sheet_name="men_bb_ncaa")
bb_22=bb[bb.year==2023]
bb_23=bb[bb.year==2024]
bb_24=bb[bb.year==2025]

In [76]:
init_ratings=1500
s=400 #elo scale
home_avt = 65
shrinkage= 0.7
k= 22 #learning rate
margin_cap= 21 ##https://kenpom.com/blog/the-simplest-proof-that-overtime-data-is-useful-and-that-garbage-time-exists/

def win_prob(r_a,r_b,h_a,h_b):
    return 1/(1+10**(-((r_a+h_a)-(r_b+h_b))/s))

def margin_multiplier(score_a,score_b):
    margin=abs(score_a-score_b)
    margin=min(margin,margin_cap)
    return np.log1p(margin)

def update_rating(r_a,r_b,binary_a,binary_b,h_a,h_b,score_a,score_b):
    p_a=win_prob(r_a,r_b,h_a,h_b)
    p_b=1-p_a

    m=margin_multiplier(score_a,score_b)
    r_a_new=r_a+ k*m*(binary_a - p_a)
    r_b_new=r_b + k*m*(binary_b - p_b)

    return r_a_new, r_b_new

def rating_shrink(r):
    return init_ratings + shrinkage*(r-init_ratings)

In [52]:
bb_22=bb_22[bb_22["winner"]!='Tie']

In [77]:
team_ratings={}
for i,row in bb_22.iterrows():
    team_a=row["home team"]
    team_b=row["away team"]
    if row['neutral site']=='False':
        h_a=home_avt
        h_b=0
    else:
        h_a=0
        h_b=0
    if team_a in team_ratings:
        r_a=team_ratings[team_a]
    else:
        r_a=init_ratings
    if team_b in team_ratings:
        r_b=team_ratings[team_b]
    else:
        r_b=init_ratings
    if row["winner"]==team_a:
        binary_a=1
        binary_b=0
    else:
        binary_a=0
        binary_b=1
    r_a_update,r_b_update=update_rating(r_a,r_b,binary_a,binary_b,h_a,h_b,row["home score"],row["away score"])
    team_ratings[team_a]=r_a_update
    team_ratings[team_b]=r_b_update
    


In [78]:
team_ratings

{'North Carolina Tar Heels': np.float64(1684.4296784540388),
 'UNC Wilmington Seahawks': np.float64(1640.812794374818),
 'Houston Cougars': np.float64(1894.5157412231072),
 'Northern Colorado Bears': np.float64(1464.2943147768462),
 'Kentucky Wildcats': np.float64(1726.186965992394),
 'Howard Bison': np.float64(1627.7045999206928),
 'Baylor Bears': np.float64(1713.1483541838495),
 'Mississippi Valley State Delta Devils': np.float64(1231.4808359115073),
 'Kansas Jayhawks': np.float64(1826.0979895276428),
 'Omaha Mavericks': np.float64(1337.494949730568),
 'Duke Blue Devils': np.float64(1828.4464534406643),
 'Jacksonville Dolphins': np.float64(1418.6930611741361),
 'UCLA Bruins': np.float64(1905.3847511293238),
 'Sacramento State Hornets': np.float64(1402.498679953071),
 'Creighton Bluejays': np.float64(1788.8801202592433),
 'St. Thomas-Minnesota Tommies': np.float64(1539.1413879596698),
 'Arkansas Razorbacks': np.float64(1693.4415930823727),
 'North Dakota State Bison': np.float64(1598.

In [79]:
bb_23=bb_23[bb_23["winner"]!='Tie']
for team in team_ratings:
    team_ratings[team]=rating_shrink(team_ratings[team])
for i,row in bb_23.iterrows():
    team_a=row["home team"]
    team_b=row["away team"]
    if row['neutral site']=='False':
        h_a=home_avt
        h_b=0
    else:
        h_a=0
        h_b=0
    if team_a in team_ratings:
        r_a=team_ratings[team_a]
    else:
        r_a=init_ratings
    if team_b in team_ratings:
        r_b=team_ratings[team_b]
    else:
        r_b=init_ratings
    if row["winner"]==team_a:
        binary_a=1
        binary_b=0
    else:
        binary_a=0
        binary_b=1
    r_a_update,r_b_update=update_rating(r_a,r_b,binary_a,binary_b,h_a,h_b,row["home score"],row["away score"])
    team_ratings[team_a]=r_a_update
    team_ratings[team_b]=r_b_update
    


In [80]:
team_ratings

{'North Carolina Tar Heels': np.float64(1914.6571170603872),
 'UNC Wilmington Seahawks': np.float64(1659.4155162007792),
 'Houston Cougars': np.float64(1988.949751883408),
 'Northern Colorado Bears': np.float64(1558.1772936073708),
 'Kentucky Wildcats': np.float64(1838.4041790488184),
 'Howard Bison': np.float64(1558.8296291864772),
 'Baylor Bears': np.float64(1824.1475789344008),
 'Mississippi Valley State Delta Devils': np.float64(1082.4591870899546),
 'Kansas Jayhawks': np.float64(1776.5663431569799),
 'Omaha Mavericks': np.float64(1464.7466429422677),
 'Duke Blue Devils': np.float64(1898.3376547141988),
 'Jacksonville Dolphins': np.float64(1486.317110783276),
 'UCLA Bruins': np.float64(1653.29179547405),
 'Sacramento State Hornets': np.float64(1380.2550669631205),
 'Creighton Bluejays': np.float64(1857.2324409877926),
 'St. Thomas-Minnesota Tommies': np.float64(1615.768658511175),
 'Arkansas Razorbacks': np.float64(1610.7062544134415),
 'North Dakota State Bison': np.float64(1487.4

In [81]:
bb_24=bb_24[bb_24["winner"]!='Tie']
for team in team_ratings:
    team_ratings[team]=rating_shrink(team_ratings[team])
for i,row in bb_24.iterrows():
    team_a=row["home team"]
    team_b=row["away team"]
    if row['neutral site']=='False':
        h_a=home_avt
        h_b=0
    else:
        h_a=0
        h_b=0
    if team_a in team_ratings:
        r_a=team_ratings[team_a]
    else:
        r_a=init_ratings
    if team_b in team_ratings:
        r_b=team_ratings[team_b]
    else:
        r_b=init_ratings
    if row["winner"]==team_a:
        binary_a=1
        binary_b=0
    else:
        binary_a=0
        binary_b=1
    r_a_update,r_b_update=update_rating(r_a,r_b,binary_a,binary_b,h_a,h_b,row["home score"],row["away score"])
    team_ratings[team_a]=r_a_update
    team_ratings[team_b]=r_b_update


In [82]:
team_ratings

{'North Carolina Tar Heels': np.float64(1838.785988911473),
 'UNC Wilmington Seahawks': np.float64(1775.403937666733),
 'Houston Cougars': np.float64(2188.7245209142484),
 'Northern Colorado Bears': np.float64(1726.7116407285646),
 'Kentucky Wildcats': np.float64(1856.877937928095),
 'Howard Bison': np.float64(1374.4996799938076),
 'Baylor Bears': np.float64(1810.1028800623583),
 'Mississippi Valley State Delta Devils': np.float64(1028.2877481373525),
 'Kansas Jayhawks': np.float64(1776.5047085346011),
 'Omaha Mavericks': np.float64(1725.3038579946817),
 'Duke Blue Devils': np.float64(2135.850459077652),
 'Jacksonville Dolphins': np.float64(1557.3483817067436),
 'UCLA Bruins': np.float64(1834.014752506937),
 'Sacramento State Hornets': np.float64(1258.6982430701014),
 'Creighton Bluejays': np.float64(1878.2984685400509),
 'St. Thomas-Minnesota Tommies': np.float64(1684.977726559037),
 'Arkansas Razorbacks': np.float64(1808.0675984238655),
 'North Dakota State Bison': np.float64(1664.20

In [91]:
## Loading current season data
current=pd.read_excel("ncaa2026.xlsx")

In [69]:
def log_loss(prob,binary):
    return -(binary*np.log(prob)+(1-binary)*np.log(1-prob))

In [92]:
current=current[current["winner"]!='Tie']

In [83]:
logloss=0
for team in team_ratings:
    team_ratings[team]=rating_shrink(team_ratings[team])
for i,row in current.iterrows():
    team_a=row["home team"]
    team_b=row["away team"]
    if team_a not in team_ratings:
        team_ratings[team_a]=init_ratings
    r_a=team_ratings[team_a]
    if team_b not in team_ratings:
        team_ratings[team_b]=init_ratings
    r_b=team_ratings[team_b]
    if row["winner"]==team_a:
        binary_a=1
        binary_b=0
    else:
        binary_a=0
        binary_b=1
    if row["neutral site"]:
        h_a=0
        h_b=0
    else:
        h_a=home_avt
        h_b=0
    prob_a= win_prob(r_a,r_b,h_a,h_b)
    prob_b=1-prob_a
    logloss += log_loss(prob_a,binary_a)
    r_a_update,r_b_update=update_rating(r_a,r_b,binary_a,binary_b,h_a,h_b,row["home score"],row["away score"])
    team_ratings[team_a]=r_a_update
    team_ratings[team_b]=r_b_update
    


In [85]:
logloss/len(current)

np.float64(0.5410581570519267)

In [90]:
win_prob(team_ratings["McNeese Cowboys"],team_ratings["Stephen F. Austin Lumberjacks"],0,0)

np.float64(0.5660810831053363)

In [97]:
yesterday=current[-38:]
for i,row in yesterday.iterrows():
    team_a=row["home team"]
    team_b=row["away team"]
    if team_a not in team_ratings:
        team_ratings[team_a]=init_ratings
    r_a=team_ratings[team_a]
    if team_b not in team_ratings:
        team_ratings[team_b]=init_ratings
    r_b=team_ratings[team_b]
    if row["winner"]==team_a:
        binary_a=1
        binary_b=0
    else:
        binary_a=0
        binary_b=1
    if row["neutral site"]:
        h_a=0
        h_b=0
    else:
        h_a=home_avt
        h_b=0
    prob_a= win_prob(r_a,r_b,h_a,h_b)
    prob_b=1-prob_a
    logloss += log_loss(prob_a,binary_a)
    r_a_update,r_b_update=update_rating(r_a,r_b,binary_a,binary_b,h_a,h_b,row["home score"],row["away score"])
    team_ratings[team_a]=r_a_update
    team_ratings[team_b]=r_b_update
    


In [112]:
win_prob(team_ratings["Miami Hurricanes"],team_ratings["Louisville Cardinals"],0,0)

np.float64(0.4329961611731)

---
---
Separated season ratings and weighted average